In [8]:
from fastapi import FastAPI, UploadFile, File
from fastapi.middleware.cors import CORSMiddleware
from PIL import Image
import io
from model import classify_image
from transformers import pipeline
import torch


app = FastAPI(title="E-commerce Quality Check API")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/predict")
async def predict(file: UploadFile = File(...)):
    content = await file.read()
    image = Image.open(io.BytesIO(content)).convert("RGB")
    result = classify_image(image)
    return result


device = 0 if torch.cuda.is_available() else -1

LABELS = [
    "a complete returned product with intact packaging, ready for resale",
    "a returned product with damaged outer packaging or crushed box",
    "an empty box or package with missing product or accessories"
]

classifier = pipeline(
    task="zero-shot-image-classification",
    model="openai/clip-vit-base-patch32",
    device=device
)

def classify_image(image):
    results = classifier(image, candidate_labels=LABELS)

    output = []
    for r in results:
        output.append({
            "label": r["label"],
            "score": float(r["score"])
        })

    top_label = results[0]["label"]
    top_score = results[0]["score"]

    if "damaged" in top_label or "crushed" in top_label:
        advice = f"❌ 建议直接退款或人工复核（确信度 {top_score:.1%}）"
    elif "empty" in top_label or "missing" in top_label:
        advice = f"❌ 疑似空盒/缺件，建议人工复核（确信度 {top_score:.1%}）"
    else:
        advice = f"✅ 商品完好，建议重新上架（确信度 {top_score:.1%}）"

    return {
        "top3": output[:3],
        "advice": advice
    }

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
